In [29]:
from models.dnnet import DNNetSiamese
import torch
from PIL import Image
import numpy as np
import torchvision.transforms as T
import configs.config as config

In [ ]:
path = 'checkpoints/best_model.pth'

ckpt = torch.load(path, map_location='cuda')
state = ckpt["model_state"]

# Infer dims from saved weights so config mismatches never matter
embedding_dim = state["dnnet.attention.fc.weight"].shape[0]
num_classes   = state["arcface_head.weight"].shape[0]
model = DNNetSiamese(num_classes, [512, 256], 128)

In [59]:
model.load_state_dict(state)
model.eval().to('cuda')

DNNetSiamese(
  (dnnet): DNNet(
    (feature_extractor): FeatureExtractionModule(
      (backbone): Sequential(
        (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (4): Sequential(
          (0): Bottleneck(
            (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=T

In [60]:
def _get_transform(image_size: int = config.IMAGE_SIZE) -> T.Compose:
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
    ])

In [67]:
with torch.no_grad():
    img = Image.open("2711100000003293_MUZZLE_CENTER_aug_3.jpg").convert('RGB')
    img = _get_transform()(img).unsqueeze(0).to('cuda')
    em = model.get_embedding(img)

In [68]:
em.cpu().numpy()

array([[ 0.07454184, -0.17537746, -0.14258361, -0.06158185, -0.01602929,
         0.16424574, -0.08040902,  0.04960418, -0.0274529 ,  0.01023046,
         0.11940021, -0.01326187, -0.04738136,  0.00418306, -0.11345119,
        -0.02964885,  0.06160073,  0.08587261, -0.13947244,  0.07144931,
        -0.00188263, -0.05713703, -0.1754978 , -0.08603807,  0.14109907,
        -0.01786423, -0.02916798, -0.10650972, -0.22717321,  0.01808975,
        -0.07239094,  0.0353063 , -0.0195466 ,  0.05093031, -0.04711226,
        -0.06149156,  0.03279785,  0.04470466, -0.01009415,  0.13240217,
         0.08799595, -0.12506819, -0.1148981 ,  0.21916577,  0.02206207,
        -0.1347538 ,  0.01213831,  0.02790371,  0.04023887, -0.02594972,
         0.00045265,  0.10280944, -0.11995053,  0.02686489, -0.08210991,
        -0.08414807, -0.04896089,  0.10626927,  0.15678951, -0.08433682,
         0.01340822,  0.13088301, -0.13125758,  0.06531718,  0.06607503,
         0.02466317,  0.06526645, -0.12877712, -0.0